## 2. 序列模型### 2.1 理论计算题**给定字符序列**："ababc"**一阶马尔可夫模型** + **拉普拉斯平滑（加 1 平滑）****词汇表**：{'a', 'b', 'c'}#### 统计转移频次序列 "ababc" 中所有相邻转移：| 转移 | 频次 ||------|------|| a→b  | 2    || b→a  | 1    || a→c  | 1    |即：n(a,b)=2, n(b,a)=1, n(a,c)=1词汇表大小 |V|=3，加 1 平滑后每种转移的分子 +1，分母 +3。#### 1. P('a'|'b')P('a'|'b') = (n(b,a)+1) / (n(b,*)+|V|) = (1+1) / (1+3) = 2/4 = **0.5**其中 n(b,*) 表示所有以 b 为起点的转移，即 b→a 出现 1 次，b→c 出现 0 次，总计 n(b,*)=1。#### 2. P('c'|'b')P('c'|'b') = (n(b,c)+1) / (n(b,*)+|V|) = (0+1) / (1+3) = 1/4 = **0.25**

### 2.2 编程题：preprocess_text编写一个函数 `preprocess_text(text, n)`，完成以下步骤：1. 将文本转换为小写，去除标点符号（保留字母和空格）2. 按空格分词3. 构建词汇表（按出现频率排序，分配整数 ID，从 0 开始）4. 用滑动窗口生成长度为 n 的特征序列和对应的下一个词标签5. 返回词汇表字典和 (特征列表, 标签列表)

In [ ]:
import refrom collections import Counterdef preprocess_text(text, n):    # 1. lower + remove punctuation (keep letters and spaces)    cleaned = re.sub(r"[^a-z\s]", "", text.lower())    # 2. tokenize by whitespace    tokens = cleaned.split()    # 3. vocab sorted by frequency, ID from 0    freq = Counter(tokens)    vocab = {w: i for i, (w, _) in enumerate(freq.most_common())}    # 4. sliding window for features and labels    features, labels = [], []    for i in range(len(tokens) - n + 1):        features.append(tokens[i : i + n])        next_idx = i + n        labels.append(tokens[next_idx] if next_idx < len(tokens) else None)    # 5. return vocab and (features, labels)    return vocab, (features, labels)# testtext = "The time machine"n = 2vocab, (features, labels) = preprocess_text(text, n)print("Vocab:", vocab)print("Features:", features)print("Labels:", labels)

In [ ]:
# extra testtext2 = "ababc ababc"n2 = 3vocab2, (features2, labels2) = preprocess_text(text2, n2)print("Vocab:", vocab2)print("Features:", features2)print("Labels:", labels2)

## 3. 循环神经网络### 3.1 理论计算题**线性 RNN（无偏置）**：$$h_t = W_{hh} h_{t-1} + W_{hx} x_t$$$$o_t = W_{oh} h_t$$**损失函数**：$$L = \frac{1}{2}\sum_{t=1}^{T}(o_t - y_t)^2$$#### 损失对 $W_{hh}$ 的梯度（BPTT 展开）**Step 1**：损失对 $o_t$ 的梯度：$$\frac{\partial L}{\partial o_t} = o_t - y_t$$**Step 2**：损失对 $h_t$（通过输出路径）：$$\frac{\partial L}{\partial h_t}\bigg|_{\text{output}} = W_{oh}^T (o_t - y_t)$$**Step 3**：定义总梯度 $\delta_t = \frac{\partial L}{\partial h_t}$，通过时间步反向传播：$$\delta_t = W_{oh}^T(o_t - y_t) + W_{hh}^T \delta_{t+1}$$（$\delta_{T+1} = 0$）**Step 4**：对 $W_{hh}$ 的梯度：$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \delta_t \cdot h_{t-1}^T$$#### 梯度消失/爆炸条件由递推 $\delta_t = W_{hh}^T \delta_{t+1} + \cdots$，展开后梯度涉及 $(W_{hh}^T)^{T-t}$ 的连乘。- **梯度消失**：当 $W_{hh}$ 的最大奇异值 $\sigma_{\max}(W_{hh}) < 1$ 时，梯度随时间步指数衰减。- **梯度爆炸**：当 $\sigma_{\max}(W_{hh}) > 1$ 时，梯度随时间步指数增长。- **稳定条件**：$\|W_{hh}\| \approx 1$ 时梯度较稳定。

### 3.2 编程题：RNN 前向传播与单步反向传播

In [ ]:
import numpy as npdef rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h):    h_t = np.tanh(x_t @ W_hx + h_prev @ W_hh + b_h)    return h_tdef rnn_step_backward(dh_next, cache):    x_t, h_prev, h_t, W_hx, W_hh = cache    dtanh = dh_next * (1 - h_t ** 2)    dW_hx = x_t.T @ dtanh    dW_hh = h_prev.T @ dtanh    db_h = np.sum(dtanh, axis=0)    dx_t = dtanh @ W_hx.T    dh_prev = dtanh @ W_hh.T    return dx_t, dh_prev, dW_hx, dW_hh, db_h# testnp.random.seed(42)batch_size, input_size, hidden_size = 2, 3, 4x_t = np.random.randn(batch_size, input_size)h_prev = np.random.randn(batch_size, hidden_size)W_hx = np.random.randn(input_size, hidden_size)W_hh = np.random.randn(hidden_size, hidden_size)b_h = np.zeros(hidden_size)h_t = rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h)print("h_t:\n", h_t)dh_next = np.ones_like(h_t)cache = (x_t, h_prev, h_t, W_hx, W_hh)dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_step_backward(dh_next, cache)print("\ndx_t shape:", dx_t.shape)print("dh_prev shape:", dh_prev.shape)print("dW_hx shape:", dW_hx.shape)print("dW_hh shape:", dW_hh.shape)print("db_h shape:", db_h.shape)

## 4. 高级循环神经网络### 4.1 理论计算题**深度双向 RNN**，$L$ 层，每层隐藏单元数 $H$，输入维度 $D$，输出维度 $O$。**参数计算**：每层有正向和反向两个独立的 RNN 子层，每个子层有：- 隐藏权重矩阵 $W_{hh} \in \mathbb{R}^{H \times H}$- 输入权重矩阵 $W_{xh} \in \mathbb{R}^{H \times H}$（上层输出维度为 $H$）- 偏置 $b_h \in \mathbb{R}^{H}$每个子层参数数 = $H^2 + H^2 + H = 2H^2 + H = H(2H+1)$每层（正向+反向）参数数 = $2 \times H(2H+1)$**$L$ 层总计**：$$\text{总参数} = 2LH(2H+1)$$

### 4.2 编程题：双向 RNN 编码器

In [ ]:
import torchimport torch.nn as nnclass BiRNNEncoder(nn.Module):    def __init__(self, input_dim, hidden_dim):        super().__init__()        self.hidden_dim = hidden_dim        self.rnn_fwd = nn.RNN(input_dim, hidden_dim, batch_first=False)        self.rnn_bwd = nn.RNN(input_dim, hidden_dim, batch_first=False)    def forward(self, X):        # X: (seq_len, batch, input_dim)        fwd_out, _ = self.rnn_fwd(X)        X_rev = torch.flip(X, dims=[0])        bwd_out, _ = self.rnn_bwd(X_rev)        bwd_out = torch.flip(bwd_out, dims=[0])        all_hidden = torch.cat([fwd_out, bwd_out], dim=-1)        final_hidden = torch.cat([fwd_out[-1], bwd_out[0]], dim=-1)        return all_hidden, final_hiddentorch.manual_seed(0)seq_len, batch, input_dim, hidden_dim = 5, 2, 3, 4X = torch.randn(seq_len, batch, input_dim)model = BiRNNEncoder(input_dim, hidden_dim)all_hidden, final_hidden = model(X)print("all_hidden shape:", all_hidden.shape, " (expected: [5, 2, 8])")print("final_hidden shape:", final_hidden.shape, " (expected: [2, 8])")

## 5. 嵌入向量### 5.1 理论计算题**Skip-gram 模型 + 负采样**中心词 $w_c$，上下文词 $w_o$，采样 $K$ 个负样本。**目标函数（最大化对数似然）**：$$\mathcal{L} = \log \sigma(u_o^T v_c) + \sum_{k=1}^{K} \mathbb{E}_{w_k \sim P_n(w)} [\log \sigma(-u_k^T v_c)]$$其中：- $v_c$：中心词 $w_c$ 的词向量- $u_o$：上下文词 $w_o$ 的词向量- $u_k$：第 $k$ 个负样本词向量- $\sigma$：sigmoid 函数- $P_n(w)$：噪声分布（通常取 unigram 的 $3/4$ 幂次分布）**负样本采样方法**：$$P_n(w) = \frac{f(w)^{3/4}}{\sum_{w'} f(w')^{3/4}}$$其中 $f(w)$ 是词 $w$ 在语料中的出现频率。

### 5.2 编程题：CBOW 模型（完整 softmax）

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Fclass CBOW(nn.Module):    def __init__(self, vocab_size, embed_dim):        super().__init__()        self.W = nn.Parameter(torch.randn(vocab_size, embed_dim) * 0.1)        self.W_out = nn.Parameter(torch.randn(embed_dim, vocab_size) * 0.1)    def forward(self, context_indices):        ctx_embeds = self.W[context_indices]        hidden = ctx_embeds.mean(dim=1)        logits = hidden @ self.W_out        return logitsdef cbow_loss(context_indices, target_indices, vocab_size, embed_dim):    model = CBOW(vocab_size, embed_dim)    logits = model(context_indices)    loss = F.cross_entropy(logits, target_indices)    return loss, logits# testtorch.manual_seed(42)vocab_size, embed_dim, context_size = 10, 5, 2batch_size = 3context_indices = torch.randint(0, vocab_size, (batch_size, context_size))target_indices = torch.randint(0, vocab_size, (batch_size,))loss, logits = cbow_loss(context_indices, target_indices, vocab_size, embed_dim)print("Loss:", loss.item())print("Logits shape:", logits.shape)

## 6. 注意力机制### 6.1 理论计算题**给定**：- $Q \in \mathbb{R}^{2 \times 4}$，$K \in \mathbb{R}^{3 \times 4}$，$V \in \mathbb{R}^{3 \times 5}$- $d_k = 4$**Step 1：计算得分矩阵**$$\text{score} = \frac{QK^T}{\sqrt{d_k}} = \frac{QK^T}{2}$$$QK^T$ 的形状：$(2 \times 4) \times (4 \times 3) = (2 \times 3)$**Step 2：Softmax**对得分矩阵的每一行做 softmax，得到注意力权重矩阵 $\alpha \in \mathbb{R}^{2 \times 3}$。$$\alpha_{ij} = \frac{\exp(\text{score}_{ij})}{\sum_{k=1}^{3} \exp(\text{score}_{ik})}$$**Step 3：加权求和**$$\text{output} = \alpha V \in \mathbb{R}^{2 \times 5}$$**输出矩阵形状**：$(2 \times 5)$

In [ ]:
import numpy as npnp.set_printoptions(precision=4)np.random.seed(42)Q = np.random.randn(2, 4)K = np.random.randn(3, 4)V = np.random.randn(3, 5)d_k = 4score = Q @ K.T / np.sqrt(d_k)print("Score matrix (2x3):")print(score)alpha = np.exp(score) / np.sum(np.exp(score), axis=1, keepdims=True)print("\nAttention weights (softmax):")print(alpha)output = alpha @ Vprint("\nOutput matrix (2x5):")print(output)

### 6.2 编程题：多头注意力（Multi-Head Attention）实现多头注意力的前向传播，假设 `num_heads=2`，`d_model=4`。给定输入 X（形状 `(seq_len, batch, d_model)`），分别线性投影得到 Q, K, V，对每个头计算缩放点积注意力，然后将所有头的输出拼接并经过最终线性层。返回输出（形状与输入相同）。

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Fimport mathclass MultiHeadAttention(nn.Module):    def __init__(self, d_model, num_heads):        super().__init__()        assert d_model % num_heads == 0        self.d_model = d_model        self.num_heads = num_heads        self.d_k = d_model // num_heads        self.W_q = nn.Linear(d_model, d_model)        self.W_k = nn.Linear(d_model, d_model)        self.W_v = nn.Linear(d_model, d_model)        self.W_o = nn.Linear(d_model, d_model)    def forward(self, X):        # X: (seq_len, batch, d_model)        seq_len, batch_size, _ = X.shape        Q = self.W_q(X).view(seq_len, batch_size, self.num_heads, self.d_k)        K = self.W_k(X).view(seq_len, batch_size, self.num_heads, self.d_k)        V = self.W_v(X).view(seq_len, batch_size, self.num_heads, self.d_k)        # -> (batch, num_heads, seq_len, d_k)        Q = Q.permute(1, 2, 0, 3)        K = K.permute(1, 2, 0, 3)        V = V.permute(1, 2, 0, 3)        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)        attn = F.softmax(scores, dim=-1)        attn_out = torch.matmul(attn, V)        # -> (seq_len, batch, d_model)        attn_out = attn_out.permute(2, 0, 1, 3).contiguous().view(seq_len, batch_size, self.d_model)        return self.W_o(attn_out)torch.manual_seed(42)num_heads, d_model = 2, 4seq_len, batch_size = 6, 2X = torch.randn(seq_len, batch_size, d_model)mha = MultiHeadAttention(d_model, num_heads)output = mha(X)print("Input shape:", X.shape)print("Output shape:", output.shape)